# TTD/TTM Validation Notebook

Extract and validate Time-To-Detect (TTD) logic from the 12-05-26-aarya run data. 
Organized by run, category, and fault name.

## 1. Import Required Libraries

In [37]:
import pandas as pd
import json
import os
from pathlib import Path
from pprint import pprint
import yaml

# Setup paths
BASE_DIR = Path("./data/input/12-05-26-aarya/Sequential/fault-bucketing")
FAULT_CATEGORIES_CONFIG = Path("./configs/fault_categories.json")

print(f"Base directory: {BASE_DIR}")
print(f"Fault categories config: {FAULT_CATEGORIES_CONFIG}")
print(f"Base dir exists: {BASE_DIR.exists()}")
print(f"Fault categories config exists: {FAULT_CATEGORIES_CONFIG.exists()}")

# Load fault categories config
fault_categories_map = {}
if FAULT_CATEGORIES_CONFIG.exists():
    with open(FAULT_CATEGORIES_CONFIG, 'r') as f:
        config = json.load(f)
    
    # Build reverse mapping: fault_name -> category
    for category, fault_names in config.get("categories", {}).items():
        for fault_name in fault_names:
            fault_categories_map[fault_name] = category
    
    print(f"\n✓ Loaded {len(fault_categories_map)} fault-to-category mappings from config")
else:
    print("\n✗ Warning: Fault categories config not found!")

Base directory: data\input\12-05-26-aarya\Sequential\fault-bucketing
Fault categories config: configs\fault_categories.json
Base dir exists: True
Fault categories config exists: True

✓ Loaded 10 fault-to-category mappings from config


## 2. Load Run Data from 12-05-26-aarya

In [38]:
# Explore the directory structure and debug path patterns
print("=== Exploring data directory structure ===")
if BASE_DIR.exists():
    for item in BASE_DIR.iterdir():
        if item.is_dir():
            print(f"\nDirectory: {item.name}")
            # List first few files in each subdirectory
            files = list(item.glob("**/*_metrics.json"))[:5]
            for f in files:
                print(f"  - {f.relative_to(BASE_DIR)}")
            if len(files) == 5:
                print(f"  ... and more")
        elif item.is_file() and item.suffix == '.json':
            print(f"File: {item.name}")
else:
    print("Base directory not found!")

# Debug: Show actual path structure of first metrics file
print("\n=== Debug: Actual File Path Structure ===")
if BASE_DIR.exists():
    sample_files = list(BASE_DIR.glob("**/*_metrics.json"))[:3]
    for f in sample_files:
        print(f"\nFile: {f.name}")
        print(f"Full path: {f}")
        print(f"Path parts: {f.parts}")


=== Exploring data directory structure ===

Directory: 000098ae-eac9-4042-ac11-546006172858
  - 000098ae-eac9-4042-ac11-546006172858\metrics\pod-cpu-hog_000098ae-eac9-4042-ac11-546006172858_metrics.json
  - 000098ae-eac9-4042-ac11-546006172858\metrics\pod-memory-hog_000098ae-eac9-4042-ac11-546006172858_metrics.json
  - 000098ae-eac9-4042-ac11-546006172858\metrics\pod-network-loss_000098ae-eac9-4042-ac11-546006172858_metrics.json

Directory: 0a0e9fcf-5dad-4701-8d5a-909cf67d523d
  - 0a0e9fcf-5dad-4701-8d5a-909cf67d523d\metrics\pod-cpu-hog_0a0e9fcf-5dad-4701-8d5a-909cf67d523d_metrics.json
  - 0a0e9fcf-5dad-4701-8d5a-909cf67d523d\metrics\pod-memory-hog_0a0e9fcf-5dad-4701-8d5a-909cf67d523d_metrics.json
  - 0a0e9fcf-5dad-4701-8d5a-909cf67d523d\metrics\pod-network-loss_0a0e9fcf-5dad-4701-8d5a-909cf67d523d_metrics.json

Directory: 0ac7ac9c-2747-4bd7-a928-3245515baca3
  - 0ac7ac9c-2747-4bd7-a928-3245515baca3\metrics\pod-cpu-hog_0ac7ac9c-2747-4bd7-a928-3245515baca3_metrics.json
  - 0ac7ac9c-2747

In [14]:
# Function to load metrics from files
def load_all_metrics(base_dir):
    """Load all *_metrics.json files from the directory structure
    
    Expected structure:
    data/input/12-05-26-aarya/Sequential/fault-bucketing/{run_id}/metrics/{fault_name}_metrics.json
    
    Metrics file structure:
    {
        "fault_name": "pod-cpu-hog",
        "run_id": "e23d55ae-b0b2-4d2d-8049-a75823e56304",
        "quantitative": {
            "run_id": "e23d55ae-b0b2-4d2d-8049-a75823e56304",
            "time_to_detect": 3.7,
            ...
        }
    }
    """
    metrics_by_run = {}
    
    # Find all _metrics.json files
    metrics_files = list(base_dir.glob("**/*_metrics.json"))
    print(f"Found {len(metrics_files)} metrics files")
    
    for metrics_file in metrics_files:
        try:
            with open(metrics_file, 'r') as f:
                metrics_data = json.load(f)
            
            # Extract run_id and fault_name from root level
            run_id = metrics_data.get('run_id')
            fault_name = metrics_data.get('fault_name')
            
            if run_id and fault_name:
                if run_id not in metrics_by_run:
                    metrics_by_run[run_id] = {}
                
                metrics_by_run[run_id][fault_name] = metrics_data
                print(f"✓ Loaded {fault_name} for run {run_id}")
            else:
                print(f"⚠ Skipping {metrics_file.name}: run_id={run_id}, fault_name={fault_name}")
                
        except Exception as e:
            print(f"Error loading {metrics_file}: {e}")
    
    return metrics_by_run

# Load metrics
print("\n=== Loading all metrics files ===")
metrics_data = load_all_metrics(BASE_DIR)
print(f"\n✓ Loaded metrics for {len(metrics_data)} runs")

# Show sample structure
if metrics_data:
    first_run = list(metrics_data.keys())[0]
    print(f"\nSample run: {first_run}")
    print(f"Faults in this run: {list(metrics_data[first_run].keys())}")
else:
    print("⚠ No metrics loaded!")


=== Loading all metrics files ===
Found 102 metrics files
✓ Loaded pod-cpu-hog for run e23d55ae-b0b2-4d2d-8049-a75823e56304
✓ Loaded pod-memory-hog for run e23d55ae-b0b2-4d2d-8049-a75823e56304
✓ Loaded pod-network-loss for run e23d55ae-b0b2-4d2d-8049-a75823e56304
✓ Loaded pod-cpu-hog for run da495936-aa5c-4fb1-a0a8-7cf5e2922222
✓ Loaded pod-memory-hog for run da495936-aa5c-4fb1-a0a8-7cf5e2922222
✓ Loaded pod-network-loss for run da495936-aa5c-4fb1-a0a8-7cf5e2922222
✓ Loaded unknown for run d50303e0-a1e4-4d6a-bdac-4101354dca69
✓ Loaded pod-cpu-hog for run c52aa625-cd95-43e6-9f6f-7ddf1def07ef
✓ Loaded pod-memory-hog for run c52aa625-cd95-43e6-9f6f-7ddf1def07ef
✓ Loaded pod-network-loss for run c52aa625-cd95-43e6-9f6f-7ddf1def07ef
✓ Loaded unknown for run b20eb483-0e35-4ae5-af23-dc77338cd2e6
✓ Loaded pod-cpu-hog for run b1bf3fb2-39ad-49b3-abf0-240d7ed2be20
✓ Loaded pod-memory-hog for run b1bf3fb2-39ad-49b3-abf0-240d7ed2be20
✓ Loaded pod-network-loss for run b1bf3fb2-39ad-49b3-abf0-240d7e

## 3. Extract TTD by Fault and Category

In [39]:
# Function to extract TTD from metrics
def extract_ttd_by_category(metrics_data, fault_category_map):
    """Extract TTD values organized by category using fault_categories config
    
    - Fault name: from root level (metrics_data['fault_name'])
    - Category: from config map only (fault_category_map[fault_name])
    - TTD: from quantitative.time_to_detect
    """
    ttd_structure = {}
    
    for run_id, faults in metrics_data.items():
        ttd_structure[run_id] = {}
        
        for fault_name, fault_metrics in faults.items():
            # Get category from config map only
            category = fault_category_map.get(fault_name, None)
            
            # Extract TTD value from quantitative section
            ttd_value = None
            if isinstance(fault_metrics, dict) and 'quantitative' in fault_metrics:
                quant = fault_metrics['quantitative']
                ttd_value = quant.get('time_to_detect')
            
            if category:
                if category not in ttd_structure[run_id]:
                    ttd_structure[run_id][category] = {}
                ttd_structure[run_id][category][fault_name] = ttd_value
            else:
                if 'Unknown' not in ttd_structure[run_id]:
                    ttd_structure[run_id]['Unknown'] = {}
                ttd_structure[run_id]['Unknown'][fault_name] = ttd_value
    
    return ttd_structure

print("Extracting TTD values by category...")
ttd_by_category = extract_ttd_by_category(metrics_data, fault_categories_map)
print(f"Extraction complete. Runs extracted: {len(ttd_by_category)}")

if ttd_by_category:
    print("\n=== Sample TTD Structure (First Run) ===")
    first_run = list(ttd_by_category.keys())[0]
    print(f"Run: {first_run}")
    pprint(ttd_by_category[first_run])
else:
    print("⚠ No TTD data extracted!")

Extracting TTD values by category...
Extraction complete. Runs extracted: 38

=== Sample TTD Structure (First Run) ===
Run: e23d55ae-b0b2-4d2d-8049-a75823e56304
{'network_fault': {'pod-network-loss': None},
 'resource_fault': {'pod-cpu-hog': 3.7, 'pod-memory-hog': 23.91}}


In [40]:
print("\n" + "="*80)
print("COMPLETE TTD DICTIONARY (run -> category -> fault -> ttd_value)")
print("="*80)
pprint(ttd_by_category, width=120, depth=5)


COMPLETE TTD DICTIONARY (run -> category -> fault -> ttd_value)
{'000098ae-eac9-4042-ac11-546006172858': {'network_fault': {'pod-network-loss': None},
                                          'resource_fault': {'pod-cpu-hog': 4.1, 'pod-memory-hog': None}},
 '0a0e9fcf-5dad-4701-8d5a-909cf67d523d': {'network_fault': {'pod-network-loss': None},
                                          'resource_fault': {'pod-cpu-hog': 3.73, 'pod-memory-hog': 9.92}},
 '0ac7ac9c-2747-4bd7-a928-3245515baca3': {'network_fault': {'pod-network-loss': None},
                                          'resource_fault': {'pod-cpu-hog': 3.46, 'pod-memory-hog': 13.11}},
 '0dd90da1-dc5e-40f5-a4cb-134ba6cd8ddd': {'network_fault': {'pod-network-loss': None},
                                          'resource_fault': {'pod-cpu-hog': 4.87, 'pod-memory-hog': 2.12}},
 '120e514b-30d9-4a5e-978a-7d9b2518d172': {'network_fault': {'pod-network-loss': None},
                                          'resource_fault': {'pod-cp

## 4. Organize TTD into Final Nested Dictionary Structure

Structure: `run -> category -> fault_name -> ttd_value`

In [41]:
print("=== Complete TTD Structure (run -> category -> fault -> ttd_value) ===\n")

ttd_summary = {}
for run_num in sorted(ttd_by_category.keys()):
    run_data = ttd_by_category[run_num]
    ttd_summary[run_num] = {}
    
    for category in run_data:
        ttd_summary[run_num][category] = {}
        for fault_name, ttd_val in run_data[category].items():
            ttd_summary[run_num][category][fault_name] = {
                'ttd_mean': ttd_val,
                'ttd_seconds': f"{ttd_val:.2f}s" if ttd_val else "N/A"
            }

for i, (run_num, run_data) in enumerate(list(ttd_summary.items())[:3]):
    print(f"{'='*60}")
    print(f"Run: {run_num}")
    print(f"{'='*60}")
    for category, faults in run_data.items():
        print(f"\n  {category}:")
        for fault_name, ttd_info in faults.items():
            print(f"    {fault_name}: {ttd_info['ttd_seconds']}")

print(f"\n... ({len(ttd_summary)} total runs)")

print(f"\n\n=== Summary Statistics ===")
print(f"Total runs: {len(ttd_by_category)}")
total_faults = sum(len(faults) for run in ttd_by_category.values() for faults in run.values())
print(f"Total fault entries: {total_faults}")

=== Complete TTD Structure (run -> category -> fault -> ttd_value) ===

Run: 000098ae-eac9-4042-ac11-546006172858

  resource_fault:
    pod-cpu-hog: 4.10s
    pod-memory-hog: N/A

  network_fault:
    pod-network-loss: N/A
Run: 0a0e9fcf-5dad-4701-8d5a-909cf67d523d

  resource_fault:
    pod-cpu-hog: 3.73s
    pod-memory-hog: 9.92s

  network_fault:
    pod-network-loss: N/A
Run: 0ac7ac9c-2747-4bd7-a928-3245515baca3

  resource_fault:
    pod-cpu-hog: 3.46s
    pod-memory-hog: 13.11s

  network_fault:
    pod-network-loss: N/A

... (38 total runs)


=== Summary Statistics ===
Total runs: 38
Total fault entries: 102


## 5. Define Hardcoded SLA Thresholds (Configurable)

In [42]:
# Load SLA thresholds from hypothesis_framework/data/groundtruth
groundtruth_dir = Path("./hypothesis_framework/data/groundtruth/kubernetes")

def load_sla_from_groundtruth(groundtruth_base_dir, fault_names):
    """
    Load SLA thresholds from groundtruth files for each fault.
    
    Expected structure:
    hypothesis_framework/data/groundtruth/kubernetes/{fault_name}/ground_truth.yaml
    
    SLA structure in YAML:
    sla:
      time_to_detect:
        threshold: 120
        unit: "seconds"
      time_to_mitigate:
        threshold: 300
        unit: "seconds"
    """
    sla_dict = {}
    
    for fault_name in fault_names:
        fault_dir = groundtruth_base_dir / fault_name
        groundtruth_file = fault_dir / "ground_truth.yaml"
        
        if groundtruth_file.exists():
            try:
                with open(groundtruth_file, 'r') as f:
                    groundtruth_data = yaml.safe_load(f)
                
                if 'ground_truth' in groundtruth_data and 'sla' in groundtruth_data['ground_truth']:
                    sla = groundtruth_data['ground_truth']['sla']
                    
                    ttd_threshold = sla.get('time_to_detect', {}).get('threshold')
                    ttm_threshold = sla.get('time_to_mitigate', {}).get('threshold')
                    
                    sla_dict[fault_name] = {
                        'ttd_sla_seconds': ttd_threshold,
                        'ttm_sla_seconds': ttm_threshold,
                        'category': fault_categories_map.get(fault_name)
                    }
                    print(f"✓ Loaded SLA for {fault_name}: TTD={ttd_threshold}s, TTM={ttm_threshold}s")
                else:
                    sla_dict[fault_name] = None
                    print(f"⚠ No SLA found in {fault_name}")
                    
            except Exception as e:
                sla_dict[fault_name] = None
                print(f"✗ Error loading SLA for {fault_name}: {e}")
        else:
            sla_dict[fault_name] = None
            print(f"⚠ No groundtruth file for {fault_name}")
    
    return sla_dict

# Load SLA for all known faults
print("=== Loading SLA Thresholds from Groundtruth ===\n")
sla_thresholds = load_sla_from_groundtruth(groundtruth_dir, fault_categories_map.keys())

print("\n=== SLA Thresholds Loaded ===\n")
for fault_name in sorted(fault_categories_map.keys()):
    sla_info = sla_thresholds.get(fault_name)
    if sla_info:
        print(f"{fault_name:30} | TTD: {sla_info['ttd_sla_seconds']:3}s | TTM: {sla_info['ttm_sla_seconds']:3}s")
    else:
        print(f"{fault_name:30} | SLA: None (not in groundtruth)")


=== Loading SLA Thresholds from Groundtruth ===

✓ Loaded SLA for node-restart: TTD=30s, TTM=300s
✓ Loaded SLA for pod-delete: TTD=60s, TTM=120s
✓ Loaded SLA for pod-dns-error: TTD=60s, TTM=180s
✓ Loaded SLA for pod-network-corruption: TTD=240s, TTM=420s
✓ Loaded SLA for pod-network-loss: TTD=180s, TTM=360s
✓ Loaded SLA for pod-network-rate-limit: TTD=180s, TTM=360s
✓ Loaded SLA for disk-fill: TTD=60s, TTM=180s
✓ Loaded SLA for pod-autoscaler: TTD=120s, TTM=180s
✓ Loaded SLA for pod-cpu-hog: TTD=120s, TTM=300s
✓ Loaded SLA for pod-memory-hog: TTD=90s, TTM=240s

=== SLA Thresholds Loaded ===

disk-fill                      | TTD:  60s | TTM: 180s
node-restart                   | TTD:  30s | TTM: 300s
pod-autoscaler                 | TTD: 120s | TTM: 180s
pod-cpu-hog                    | TTD: 120s | TTM: 300s
pod-delete                     | TTD:  60s | TTM: 120s
pod-dns-error                  | TTD:  60s | TTM: 180s
pod-memory-hog                 | TTD:  90s | TTM: 240s
pod-network-corr

In [43]:
## Fault-Level TTD Normalization (SLA-Aware with Continuous Decay)

def normalize_speed_sla_aware(mean_seconds, sla_threshold, percentile=None):
    """
    SLA-aware normalization with continuous monotonic decay.
    - Within SLA: 1.0 (at 0s) → 0.15 (at threshold)
    - Over SLA: 0.15 (at threshold) → 0 (at 1.5x threshold)
    
    Args:
        mean_seconds: Actual TTD/TTM in seconds (0 = no detection/failure)
        sla_threshold: SLA threshold in seconds
        percentile: Optional percentile info (unused, for compatibility)
    
    Returns:
        score: Normalized score 0-1 (higher is better)
               0 = no detection (failure) or severe overage
               1 = perfect detection near 0s
    """
    # 0 means no detection (failure) → score = 0
    if mean_seconds is None or mean_seconds == 0:
        return 0.0
    
    ratio = mean_seconds / sla_threshold
    
    if ratio <= 1.0:
        # Within SLA: linear scale from ~1.0 to 0.15
        return 1 - ratio * 0.85
    else:
        # Over SLA: decay from 0.15 to 0
        # slope = -0.3 reaches 0 at 1.5x threshold
        overage_ratio = ratio - 1.0
        return max(0, 0.15 - overage_ratio * 0.3)


def normalize_ttd_with_fault_sla(fault_name, ttd_value, sla_dict):
    """
    Normalize TTD using fault-specific SLA from groundtruth.
    Uses SLA-aware continuous decay normalization.
    
    Args:
        fault_name: Name of the fault (e.g., 'pod-cpu-hog')
        ttd_value: Time-to-detect in seconds (0 or None = no detection = failure)
        sla_dict: Dictionary with fault-level SLAs (loaded from groundtruth)
    
    Returns:
        score: Normalized score 0-1 (higher is better), or None if SLA not found
        sla_seconds: The SLA used for normalization, or None
    
    Logic (SLA-Aware Continuous Decay):
        - If no SLA: return (None, None)
        - If ttd is 0 or None: return (0.0, sla) [no detection = failure]
        - Within SLA (ratio ≤ 1.0): score = 1 - ratio * 0.85
          → At 0s: score = 1.0 (perfect, but 0 is treated as failure)
          → At SLA: score = 0.15 (barely acceptable)
        - Over SLA (ratio > 1.0): score = max(0, 0.15 - (ratio - 1.0) * 0.3)
          → At SLA: score = 0.15
          → At 1.5x SLA: score = 0.0 (failure)
          → Beyond 1.5x: score = 0.0 (complete failure)
    """
    if fault_name not in sla_dict or sla_dict[fault_name] is None:
        return None, None
    
    sla_seconds = sla_dict[fault_name].get('ttd_sla_seconds')
    
    if sla_seconds is None or sla_seconds == 0:
        return None, sla_seconds
    
    # If ttd is 0 or None: no detection occurred → score = 0 (failure)
    if ttd_value is None or ttd_value == 0:
        return 0.0, sla_seconds
    
    # Use new SLA-aware normalization with continuous decay
    score = normalize_speed_sla_aware(ttd_value, sla_seconds)
    
    return score, sla_seconds

# Example usage comparing old vs new normalization
print("\n" + "="*80)
print("FAULT-LEVEL TTD NORMALIZATION (SLA-Aware with Continuous Decay)")
print("="*80 + "\n")

examples = [
    ("pod-cpu-hog", None),  # No detection: score = 0 (failure)
    ("pod-cpu-hog", 0),     # No detection: score = 0 (failure)
    ("pod-cpu-hog", 60),    # Good: well under SLA
    ("pod-cpu-hog", 120),   # Threshold: at SLA
    ("pod-cpu-hog", 180),   # Bad: 1.5x SLA
    ("pod-cpu-hog", 240),   # Failure: 2x SLA
    ("pod-network-loss", 30),   # Good
    ("pod-network-loss", 60),   # At SLA
    ("pod-network-loss", 90),   # Over SLA
]

print(f"{'Fault':<25} | {'TTD':<6} | {'SLA':<4} | {'Ratio':<6} | {'Score':<6} | {'Status':<15}")
print("-" * 90)

for fault, ttd in examples:
    score, sla = normalize_ttd_with_fault_sla(fault, ttd, sla_thresholds)
    if sla:
        if ttd is None or ttd == 0:
            ratio_str = "N/A"
            status = "✗ NOT DETECTED"
        else:
            ratio = ttd / sla
            ratio_str = f"{ratio:.2f}"
            status = "✓ PERFECT" if score >= 0.9 else "✓ GOOD" if score >= 0.5 else "⚠ WARN" if score >= 0.15 else "✗ FAIL"
        
        ttd_str = "No Detect" if (ttd is None or ttd == 0) else str(ttd)
        print(f"{fault:<25} | {ttd_str:<6} | {sla:<4} | {ratio_str:<6} | {score:<6.3f} | {status:<15}")
    else:
        print(f"{fault:<25} | {ttd!s:<6} | SLA: None (no groundtruth)")



FAULT-LEVEL TTD NORMALIZATION (SLA-Aware with Continuous Decay)

Fault                     | TTD    | SLA  | Ratio  | Score  | Status         
------------------------------------------------------------------------------------------
pod-cpu-hog               | No Detect | 120  | N/A    | 0.000  | ✗ NOT DETECTED 
pod-cpu-hog               | No Detect | 120  | N/A    | 0.000  | ✗ NOT DETECTED 
pod-cpu-hog               | 60     | 120  | 0.50   | 0.575  | ✓ GOOD         
pod-cpu-hog               | 120    | 120  | 1.00   | 0.150  | ⚠ WARN         
pod-cpu-hog               | 180    | 120  | 1.50   | 0.000  | ✗ FAIL         
pod-cpu-hog               | 240    | 120  | 2.00   | 0.000  | ✗ FAIL         
pod-network-loss          | 30     | 180  | 0.17   | 0.858  | ✓ GOOD         
pod-network-loss          | 60     | 180  | 0.33   | 0.717  | ✓ GOOD         
pod-network-loss          | 90     | 180  | 0.50   | 0.575  | ✓ GOOD         


## 6. Verify and Validate TTD Values

In [44]:
print("=== TTD Validation & Statistics ===\n")

all_ttd_values = []
missing_ttd_count = 0
valid_ttd_count = 0

for run_num, run_data in ttd_by_category.items():
    for category, faults in run_data.items():
        for fault_name, ttd_val in faults.items():
            if ttd_val is None:
                missing_ttd_count += 1
            else:
                all_ttd_values.append(ttd_val)
                valid_ttd_count += 1

print(f"Valid TTD values: {valid_ttd_count}")
print(f"Missing TTD values: {missing_ttd_count}")
print(f"Total entries: {valid_ttd_count + missing_ttd_count}")

if all_ttd_values:
    print(f"\n=== TTD Statistics ===")
    print(f"Min TTD: {min(all_ttd_values):.2f}s")
    print(f"Max TTD: {max(all_ttd_values):.2f}s")
    print(f"Mean TTD: {sum(all_ttd_values) / len(all_ttd_values):.2f}s")
    print(f"Median TTD: {sorted(all_ttd_values)[len(all_ttd_values)//2]:.2f}s")

print(f"\n=== TTD Breakdown by Category ===")
category_stats = {}
for run_num, run_data in ttd_by_category.items():
    for category, faults in run_data.items():
        if category not in category_stats:
            category_stats[category] = []
        
        for fault_name, ttd_val in faults.items():
            if ttd_val is not None:
                category_stats[category].append(ttd_val)

for category in sorted(category_stats.keys()):
    values = category_stats[category]
    if values:
        print(f"\n{category}:")
        print(f"  Count: {len(values)}")
        print(f"  Avg: {sum(values) / len(values):.2f}s")
        print(f"  Min: {min(values):.2f}s")
        print(f"  Max: {max(values):.2f}s")

=== TTD Validation & Statistics ===

Valid TTD values: 60
Missing TTD values: 42
Total entries: 102

=== TTD Statistics ===
Min TTD: 0.00s
Max TTD: 69.51s
Mean TTD: 12.89s
Median TTD: 6.74s

=== TTD Breakdown by Category ===

Unknown:
  Count: 8
  Avg: 2.27s
  Min: 0.00s
  Max: 13.60s

network_fault:
  Count: 1
  Avg: 14.56s
  Min: 14.56s
  Max: 14.56s

resource_fault:
  Count: 51
  Avg: 14.52s
  Min: 0.44s
  Max: 69.51s


## 7. Export Data for Further Analysis

In [45]:
output_file = Path("./ttd_validation_export.json")

export_data = {
    "ttd_by_run_category_fault": ttd_by_category,
    "sla_thresholds": sla_thresholds,
    "statistics": {
        "total_runs": len(ttd_by_category),
        "total_fault_entries": valid_ttd_count + missing_ttd_count,
        "valid_ttd_entries": valid_ttd_count,
        "missing_ttd_entries": missing_ttd_count,
    }
}

with open(output_file, 'w') as f:
    json.dump(export_data, f, indent=2)

print(f"✓ Exported TTD validation data to: {output_file}")
print(f"  File size: {output_file.stat().st_size / 1024:.2f} KB")

print("\n=== Final Summary ===")
print(f"TTD Structure: run -> category -> fault_name -> ttd_value (in seconds)")
print(f"Total runs processed: {len(ttd_by_category)}")
print(f"SLA categories found: {len(sla_thresholds)}")
print(f"Data ready for TTD/TTM normalization validation")

✓ Exported TTD validation data to: ttd_validation_export.json
  File size: 9.13 KB

=== Final Summary ===
TTD Structure: run -> category -> fault_name -> ttd_value (in seconds)
Total runs processed: 38
SLA categories found: 10
Data ready for TTD/TTM normalization validation


In [46]:
## Aggregation: Fault → Category → Cross-Run

def normalize_and_aggregate(ttd_dict, sla_dict, fault_category_map):
    """
    Normalize TTD scores by fault, then aggregate to category and cross-run levels.
    
    Structure:
    1. Normalize each fault's TTD using fault-level SLA
    2. Aggregate per-category (mean of fault scores within category, per run)
    3. Aggregate per-category across runs (mean of all runs)
    
    Returns:
        {
            "normalized_scores": {run_id: {category: {fault: score, ...}, ...}, ...},
            "category_scores_per_run": {run_id: {category: mean_score, ...}, ...},
            "category_scores_overall": {category: {mean, min, max, std, count}, ...}
        }
    """
    
    normalized = {}
    category_by_run = {}
    category_overall = {}
    
    # Step 1: Normalize each fault score and organize by run/category
    for run_id, categories in ttd_dict.items():
        normalized[run_id] = {}
        category_by_run[run_id] = {}
        
        for category, faults in categories.items():
            normalized[run_id][category] = {}
            fault_scores = []
            
            for fault_name, ttd_value in faults.items():
                # Normalize using fault-level SLA
                score, sla = normalize_ttd_with_fault_sla(fault_name, ttd_value, sla_dict)
                
                normalized[run_id][category][fault_name] = {
                    'ttd': ttd_value,
                    'sla': sla,
                    'score': score
                }
                
                if score is not None:
                    fault_scores.append(score)
            
            # Step 2: Aggregate to category level (per run)
            if fault_scores:
                category_by_run[run_id][category] = {
                    'mean': sum(fault_scores) / len(fault_scores),
                    'min': min(fault_scores),
                    'max': max(fault_scores),
                    'count': len(fault_scores)
                }
            else:
                category_by_run[run_id][category] = {'mean': None, 'count': 0}
    
    # Step 3: Aggregate per-category across all runs
    for run_id, categories in category_by_run.items():
        for category, stats in categories.items():
            if category not in category_overall:
                category_overall[category] = []
            
            if stats['mean'] is not None:
                category_overall[category].append(stats['mean'])
    
    # Compute statistics for each category across all runs
    category_stats = {}
    for category, scores in category_overall.items():
        if scores:
            import statistics
            category_stats[category] = {
                'mean': sum(scores) / len(scores),
                'min': min(scores),
                'max': max(scores),
                'median': statistics.median(scores),
                'stdev': statistics.stdev(scores) if len(scores) > 1 else 0,
                'count_runs': len(scores),
                'count_values': sum(1 for s in scores if s is not None)
            }
    
    return {
        "normalized_scores": normalized,
        "category_scores_per_run": category_by_run,
        "category_scores_overall": category_stats
    }

# Run aggregation
print("="*80)
print("AGGREGATING NORMALIZED TTD SCORES")
print("="*80 + "\n")

aggregation = normalize_and_aggregate(ttd_by_category, sla_thresholds, fault_categories_map)

print("\n" + "="*80)
print("CATEGORY-LEVEL SCORES ACROSS ALL RUNS")
print("="*80 + "\n")

for category in sorted(aggregation["category_scores_overall"].keys()):
    stats = aggregation["category_scores_overall"][category]
    print(f"\n{category.upper()}")
    print(f"  Mean Score:  {stats['mean']:.3f}  (0.0=fail, 1.0=perfect)")
    print(f"  Median:      {stats['median']:.3f}")
    print(f"  Range:       {stats['min']:.3f} - {stats['max']:.3f}")
    print(f"  Std Dev:     {stats['stdev']:.3f}")
    print(f"  Runs:        {stats['count_runs']}")

print("\n" + "="*80)
print("PER-RUN CATEGORY SCORES (Sample: First 3 Runs)")
print("="*80 + "\n")

for i, (run_id, categories) in enumerate(list(aggregation["category_scores_per_run"].items())[:3]):
    print(f"\nRun: {run_id}")
    for category in sorted(categories.keys()):
        stats = categories[category]
        score_str = f"{stats['mean']:.3f}" if stats['mean'] is not None else "N/A"
        print(f"  {category:20} score: {score_str}  ({stats['count']} faults)")


AGGREGATING NORMALIZED TTD SCORES


CATEGORY-LEVEL SCORES ACROSS ALL RUNS


NETWORK_FAULT
  Mean Score:  0.030  (0.0=fail, 1.0=perfect)
  Median:      0.000
  Range:       0.000 - 0.931
  Std Dev:     0.167
  Runs:        31

RESOURCE_FAULT
  Mean Score:  0.715  (0.0=fail, 1.0=perfect)
  Median:      0.834
  Range:       0.000 - 0.975
  Std Dev:     0.246
  Runs:        31

PER-RUN CATEGORY SCORES (Sample: First 3 Runs)


Run: e23d55ae-b0b2-4d2d-8049-a75823e56304
  network_fault        score: 0.000  (1 faults)
  resource_fault       score: 0.874  (2 faults)

Run: da495936-aa5c-4fb1-a0a8-7cf5e2922222
  network_fault        score: 0.000  (1 faults)
  resource_fault       score: 0.484  (2 faults)

Run: d50303e0-a1e4-4d6a-bdac-4101354dca69
  Unknown              score: N/A  (0 faults)
